# Basic Nvidia Inferencing



Install dependencies. Nvidia endpoints package drags the core langchain package and its dependencies so I am not calling it explicitly. To implement 'memory' I need also langchain-community

In [ ]:
%pip install -U langchain-nvidia-ai-endpoints langchain_community --quiet

Import libraries and Nvidia key

In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [3]:
from google.colab import userdata
NVIDIA_API_KEY = userdata.get('apikey')

Initializing the model

In [4]:
llm = ChatNVIDIA(
    model="meta/llama3-8b-instruct",
    api_key = NVIDIA_API_KEY,
    temperature = 0.5,
    max_completion_tokens = 1024
    )

Create a prompt to use with the chain, but this time we will inject the history

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}")
])

Create the chain as usual

In [6]:
chain = prompt | llm

Define a session store and function to retrieve specific converstation. Here is is just an in-memory dictionary but in a production, this could be a database

In [7]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

Now we need to wrap the chain with 'RunnableWithMessageHistory'. Attention to "input" and "history" as they need to match their definition in the prompt template

In [12]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

Now we invoke with the wrapped chain. We need to pass config which contains a 'session_id'. I am creating an ID manually, but in production we would have to manage the creation of unique session ids

In [13]:
config = {"configurable": {"session_id": "user_123"}}

response1 = chain_with_history.invoke(
    {"input": "My name is Tom. What is the color of the sky?"},
    config=config
)
print(response1.content)

Nice to meet you, Tom! The color of the sky can appear different depending on the time of day and atmospheric conditions. However, during the daytime when the sun is overhead, the sky typically appears blue to our eyes. This is because the Earth's atmosphere scatters sunlight in all directions and shorter (blue) wavelengths are scattered more than longer (red) wavelengths, giving the sky its blue hue.


Examine the 'store' variable

In [15]:
store['user_123']

InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Tom. What is the color of the sky?', additional_kwargs={}, response_metadata={}), AIMessage(content="Nice to meet you, Tom! The color of the sky can appear different depending on the time of day and atmospheric conditions. However, during the daytime when the sun is overhead, the sky typically appears blue to our eyes. This is because the Earth's atmosphere scatters sunlight in all directions and shorter (blue) wavelengths are scattered more than longer (red) wavelengths, giving the sky its blue hue.", additional_kwargs={}, response_metadata={'role': 'assistant', 'content': "Nice to meet you, Tom! The color of the sky can appear different depending on the time of day and atmospheric conditions. However, during the daytime when the sun is overhead, the sky typically appears blue to our eyes. This is because the Earth's atmosphere scatters sunlight in all directions and shorter (blue) wavelengths are scattered more tha

In [16]:
response2 = chain_with_history.invoke(
    {"input": "Can you remember my name?"},
    config=config)
print(response2.content)

Tom! Yes, I can remember your name. I'm designed to keep track of conversations and remember individual names and details. So, feel free to ask me anything, and I'll do my best to assist you!
